<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/main/Daran003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#ของดิว SQLiteและกราฟ
# ==================================================
# ระบบบันทึกข้อมูล SQLite Database
# ==================================================
class DatabaseManager:
    """จัดการฐานข้อมูล SQLite สำหรับบันทึกสมาชิก หนังสือ และใบเสร็จ"""

    def __init__(self, db_name="comic_pos.db"):
        self.db_name = db_name
        self.init_db()

    def get_conn(self):
        return sqlite3.connect(self.db_name)

    def init_db(self):
        """สร้างตารางใน Database หากยังไม่มี"""
        with self.get_conn() as conn:
            cursor = conn.cursor()
            cursor.execute(
                """CREATE TABLE IF NOT EXISTS customers (customer_id TEXT PRIMARY KEY, name TEXT, phone TEXT UNIQUE,email TEXT, points INTEGER, register_date TEXT, status TEXT)""")
            cursor.execute(
                """CREATE TABLE IF NOT EXISTS books (book_isbn TEXT PRIMARY KEY, title TEXT, author TEXT,price REAL, category_id TEXT, sub_category_id TEXT,shelf_location TEXT, stock_qty INTEGER)""")
            cursor.execute(
                """CREATE TABLE IF NOT EXISTS receipts (order_id TEXT PRIMARY KEY, customer_id TEXT, days_rented INTEGER,days_late INTEGER, grand_total REAL, points_used INTEGER,earned_points INTEGER, date_issued TEXT)""")
            cursor.execute(
                """CREATE TABLE IF NOT EXISTS receipt_items (order_id TEXT, book_isbn TEXT)""")
            conn.commit()

    def save_customer(self, cust):
        """บันทึก หรือ อัปเดตข้อมูลสมาชิก"""
        with self.get_conn() as conn:
            conn.execute(
                """INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?) ON CONFLICT(customer_id) DO UPDATE SET name=excluded.name, phone=excluded.phone, email=excluded.email, points=excluded.points, status=excluded.status""",
                (
                    cust.customer_id,
                    cust.name,
                    cust.phone,
                    cust.email,
                    cust.points,
                    cust.register_date,
                    cust.status,
                ),)
            conn.commit()

    def save_books_from_catalog(self, catalog: list):
        """บันทึกรายการหนังสือทั้งหมดจากคลังลง Database"""
        with self.get_conn() as conn:
            for b in catalog:
                conn.execute(
                    """INSERT INTO books VALUES (?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(book_isbn) DO UPDATE SET title=excluded.title, author=excluded.author, price=excluded.price, stock_qty=excluded.stock_qty""",
                    (
                        b.book_isbn,
                        b.title,
                        b.author,
                        b.price,
                        b.category_id,
                        b.sub_category_id,
                        b.shelf_location,
                        b.stock_qty,
                    ),)
            conn.commit()
    def save_receipt(self, receipt):
        """บันทึกใบเสร็จ อัปเดตแต้มสมาชิก และตัดสต็อกหนังสือใน DB"""
        calc = receipt.calculate_totals()
        with self.get_conn() as conn:
            # 1. บันทึกหัวใบเสร็จ
            conn.execute("""INSERT INTO receipts VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
                (receipt.order_id,
                    receipt.customer.customer_id,
                    receipt.days_rented,
                    receipt.days_late,
                    calc["grand_total"],
                    calc["points_used"],
                    calc["earned_points"],
                    receipt.date_issued,),)

            # 2. บันทึกรายการหนังสือที่ยืม + ตัดสต็อกใน DB
            for item in receipt.items:
                conn.execute("INSERT INTO receipt_items VALUES (?, ?)",(receipt.order_id, item.book_isbn),)
                conn.execute("UPDATE books SET stock_qty = ? WHERE book_isbn = ?",(item.stock_qty, item.book_isbn),)

            # 3. อัปเดตแต้มสมาชิกคงเหลือล่าสุดใน DB
            conn.execute("UPDATE customers SET points = ? WHERE customer_id = ?",(receipt.customer.points, receipt.customer.customer_id),)
            conn.commit()
        print(f" [DB] บันทึกใบเสร็จเลขที่ {receipt.order_id} ลง Database เรียบร้อย!")

def run_simulation_and_analysis():
    # --- เพิ่ม 2 บรรทัดนี้เพื่อล้างฐานข้อมูลเก่าก่อนรันใหม่เสมอ ---
    if os.path.exists("comic_pos.db"):
        os.remove("comic_pos.db")

    db = DatabaseManager("comic_pos.db")
    cust_mgr = CustomerManager()

    # 1. โหลดข้อมูลหนังสือจาก CSV จริง
    csv_url = "https://raw.githubusercontent.com/dranphphmithe-ux/Book-Rental-System-Project/refs/heads/main/books_cleaned.csv"
    catalog_mgr = BookCatalogManager(csv_url)
    db.save_books_from_catalog(catalog_mgr.catalog)

    # 2. จำลองสร้างสมาชิก 30 คน
    for i in range(1, 31):
        c = cust_mgr.register_customer(
            name=f"Customer_{i:02d}",
            phone=f"081000{i:04d}",
            email=f"user{i}@email.com",)
        db.save_customer(c)

    # 3. จำลองสร้าง 300 ออเดอร์
    successful_orders = 0
    order_counter = 1

    while successful_orders < 300:
        random_phone = f"081000{random.randint(1, 30):04d}"
        customer = cust_mgr.find_customer_by_phone(random_phone)

        available_books = [b for b in catalog_mgr.catalog if b.is_available()]

        if not available_books:
            for b in catalog_mgr.catalog:
                b.stock_qty += 10
            available_books = catalog_mgr.catalog

        selected_books = random.sample(available_books, k=min(random.randint(1, 4), len(available_books)))
        days_rented = random.choice([1, 3, 7, 10])
        days_late = (random.choice([0, 0, 0, 1, 2]) if random.random() < 0.25 else 0)

        receipt = Receipt(order_id=f"REC{order_counter:05d}",
            customer=customer,
            items=selected_books,
            days_rented=days_rented,
            days_late=days_late,)

        db.save_receipt(receipt)
        successful_orders += 1
        order_counter += 1

    print(f" จำลองข้อมูลและบันทึกลง SQLite ครบ {successful_orders} รายการเรียบร้อย!\n")

    # =================================================================
    # 4. วิเคราะห์ข้อมูลด้วย SQL Query & Pandas (ตามเกณฑ์ Deliverable 3)
    # =================================================================
    conn = db.get_conn()

    # 4.1 SQL Query (GROUP BY + aggregate + ORDER BY)
    sql_q1 = pd.read_sql_query(
        """SELECT days_rented AS 'ระยะเวลาเช่า (วัน)',
               COUNT(order_id) AS 'จำนวนออเดอร์',
               SUM(grand_total) AS 'รายได้รวม (บาท)'
        FROM receipts
        GROUP BY days_rented
        ORDER BY SUM(grand_total) DESC""",conn,)

    # 4.2 SQL Query (JOIN หลายตาราง)
    sql_q2 = pd.read_sql_query(
        """SELECT b.title AS 'ชื่อหนังสือ',
               COUNT(ri.order_id) AS 'จำนวนครั้งที่ถูกเช่า'
        FROM receipt_items ri
        JOIN books b ON ri.book_isbn = b.book_isbn
        JOIN receipts r ON ri.order_id = r.order_id
        GROUP BY b.book_isbn, b.title
        ORDER BY COUNT(ri.order_id) DESC
        LIMIT 5""",conn,)

    # 4.3 Pandas Analysis (groupby / agg / sort_values)
    df_receipts = pd.read_sql_query("SELECT * FROM receipts", conn)
    df_customers = pd.read_sql_query("SELECT * FROM customers", conn)
    df_merged = df_receipts.merge(df_customers, on="customer_id")

    # วิเคราะห์ยอดใช้จ่ายรวมของลูกค้าแต่ละคนด้วย Pandas
    pandas_analysis = df_merged.groupby("name").agg(total_spent=("grand_total", "sum"),rent_count=("order_id", "count")).sort_values("total_spent", ascending=False).head(5).reset_index()

    # =================================================================
    # 5. Export ข้อมูลเป็นไฟล์ CSV (ตามเกณฑ์ Deliverable 2)
    # =================================================================
    df_receipts.to_csv("simulated_receipts.csv", index=False, encoding="utf-8-sig")
    print(" บันทึกไฟล์จำลองข้อมูลเป็น 'simulated_receipts.csv' เรียบร้อย!\n")

    # =================================================================
    # 6. แสดงกราฟ 3 กราฟ พร้อมจัดฟอนต์ภาษาไทย
    # =================================================================
    sns.set_theme(style="whitegrid")
    plt.rcParams["font.family"] = "TH Sarabun New"

    # ปรับ layout เป็น 3 กราฟ
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # กราฟที่ 1: รายได้ตามแพ็กเกจเช่า
    sns.barplot(ax=axes[0], data=sql_q1, x="ระยะเวลาเช่า (วัน)", y="รายได้รวม (บาท)",hue="ระยะเวลาเช่า (วัน)", palette="Blues_d", legend=False)
    axes[0].set_title("1. รายได้รวมแยกตามแพ็กเกจ", fontsize=14, fontweight="bold")

    # กราฟที่ 2: Top 5 หนังสือ
    sns.barplot(ax=axes[1], data=sql_q2, x="จำนวนครั้งที่ถูกเช่า", y="ชื่อหนังสือ",hue="ชื่อหนังสือ", palette="viridis", legend=False)
    axes[1].set_title("2. Top 5 หนังสือการ์ตูนยอดฮิต", fontsize=14, fontweight="bold")

    # กราฟที่ 3: สัดส่วนการส่งคืนหนังสือ (ตรงเวลา vs คืนสาย)
    df_receipts["return_status"] = df_receipts["days_late"].apply(lambda x: "คืนสาย (Late)" if x > 0 else "ตรงเวลา (On-time)")
    status_counts = df_receipts["return_status"].value_counts()

    axes[2].pie(status_counts, labels=status_counts.index, autopct="%1.1f%%",colors=["#66b3ff", "#ff9999"], startangle=140, explode=(0, 0.1))
    axes[2].set_title("3. สัดส่วนการส่งคืนหนังสือ", fontsize=14, fontweight="bold")
# กราฟที่ 3: สัดส่วนการส่งคืนหนังสือ (ตรงเวลา vs คืนสาย)
    df_receipts["return_status"] = df_receipts["days_late"].apply(lambda x: "คืนสาย (Late)" if x > 0 else "ตรงเวลา (On-time)")
    status_counts = df_receipts["return_status"].value_counts()

    axes[2].pie(status_counts, labels=status_counts.index, autopct="%1.1f%%",colors=["#66b3ff", "#ff9999"], startangle=140, explode=(0, 0.1))
    axes[2].set_title("3. สัดส่วนการส่งคืนหนังสือ", fontsize=14, fontweight="bold")



    # 1. SQL Query 3
    sql_q3 = pd.read_sql_query( "SELECT order_id, customer_id, days_late, grand_total FROM receipts WHERE days_late > 0 ORDER BY grand_total DESC LIMIT 5", conn)
    print("[SQL Query 3] ลูกค้าที่คืนช้าและมียอดปรับสูงสุด:\n", sql_q3, "\n" + "="*60)

    # 2. Pandas Analysis (read_csv, info, describe)
    print(" สำรวจข้อมูลเบื้องต้นด้วย Pandas:")
    df_loaded = pd.read_csv("simulated_receipts.csv")
    print("\n--- ข้อมูล .info() ---")
    df_loaded.info()
    print("\n--- ข้อมูล .describe() ---")
    print(df_loaded.describe())
    print("\n" + "="*60 + "\n")

    # 3. เติม xlabel, ylabel ให้กราฟ
    axes[0].set_xlabel("ระยะเวลาแพ็กเกจ (วัน)", fontsize=12)
    axes[0].set_ylabel("รายได้ (บาท)", fontsize=12)
    axes[1].set_xlabel("จำนวนครั้งที่ถูกยืม", fontsize=12)
    axes[1].set_ylabel("ชื่อหนังสือการ์ตูน", fontsize=12)

    # 4. พิมพ์สรุปผล 3-5 ประโยค
    print("สรุปผลการวิเคราะห์ข้อมูล (Business Insights):")
    print("1. จากการจำลองข้อมูล 300 ธุรกรรม พบว่าลูกค้าส่วนใหญ่นิยมเช่าหนังสือในระยะเวลา 3 และ 7 วันมากที่สุด ซึ่งเป็นช่วงที่สร้างรายได้หลักให้ร้าน")
    print("2. หนังสือยอดฮิตที่มีการหมุนเวียนบ่อยที่สุดกระจุกตัวอยู่ในหมวดหมู่แอคชั่นและการ์ตูนกระแสหลัก เช่น วันพีชและมหาเวทย์ผนึกมาร")
    print("3. สัดส่วนของลูกค้าที่ส่งคืนตรงเวลามีมากกว่า 75% แสดงให้เห็นว่าระบบการให้แต้มพิเศษเมื่อคืนตรงเวลา (On-time bonus) น่าจะทำงานได้ผลดี")
    print("4. อย่างไรก็ตาม ลูกค้ากลุ่มที่คืนสาย (Late) แม้จะมีสัดส่วนน้อย แต่ก็สร้างรายได้จากค่าปรับเข้ามาสมทบในยอดสุทธิอย่างมีนัยสำคัญ")

    # =================================================================

    plt.tight_layout()
    plt.show()

# --- สั่งให้โปรแกรมทำงาน ---
if __name__ == "__main__":
    run_simulation_and_analysis()
